# Track 4: обучение и тест

ЭМГ браслета (16 каналов, 2 кГц, окно 5 с) → 20 углов суставов кисти в радианах.
Метрика лидерборда: средняя по людям ошибка угла в градусах (`subject_mean`), меньше лучше.
`test` здесь = разогрев (новые люди × новые движения); его метки публичны, поэтому число ориентировочное.

Отправка одной командой из папки `track4_emg`: `python -m emg_pose.make_submission --seeds 3 --epochs 40`.

In [ ]:
import sys; sys.path.append("..")
import pandas as pd, torch
from emg_pose import data
from emg_pose.nets import PoseEnsemble
from emg_pose.train import evaluate, train

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SYNTHETIC = False  # True: маленькие фейковые данные, чтобы проверить код без скачивания
if SYNTHETIC:
    loaders, meta = data.synthetic()
else:
    loaders, meta = data.loaders("../data", batch_size=64, num_workers=8)
EPOCHS, SEEDS = (3, 2) if SYNTHETIC else (40, 3)
print(meta, {name: len(l.dataset) for name, l in loaders.items()})

## Одна модель
Лучшая эпоха выбирается по `subject_mean` на val (новые люди).

In [ ]:
net, history = train(loaders, meta, epochs=EPOCHS, device=DEVICE, seed=0)
pd.DataFrame(history).set_index("epoch")[["loss", "subject_mean"]].plot(title="градусы: train loss и val subject_mean");

In [ ]:
results = {"одна модель": {split: evaluate(PoseEnsemble([net]), loaders[split])["subject_mean"]
                            for split in ["val", "test"]}}
pd.DataFrame(results).T.round(2)

## Ансамбль
Ещё несколько сидов; на тесте усредняются модели и повороты браслета на −1, 0, +1 электрод.

In [ ]:
nets = [net] + [train(loaders, meta, epochs=EPOCHS, device=DEVICE, seed=s, log=lambda r: None)[0]
                for s in range(1, SEEDS)]
for name, model in [("ансамбль", PoseEnsemble(nets)),
                    ("ансамбль + повороты", PoseEnsemble(nets, tta_offsets=(-1, 0, 1)))]:
    results[name] = {split: evaluate(model, loaders[split])["subject_mean"] for split in ["val", "test"]}
print("ориентиры на test, градусы: VEMG2Pose 16.46, NeuroPose 17.5")
pd.DataFrame(results).T.round(2)

## ZIP для Codabench

In [ ]:
from emg_pose.export import check_zip, save_submission
zip_path = save_submission(PoseEnsemble(nets, tta_offsets=(-1, 0, 1)), "../submissions/posenet")
check_zip(zip_path, n_times=meta["n_times"])